# Ensemble Retrieval — Combining Multiple Retrieval Strategies

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/ensemble_retrieval.ipynb)

In the previous notebook **[fusion_retrieval.ipynb](fusion_retrieval.ipynb)** we built a hybrid BM25 + dense pipeline and merged results with Reciprocal Rank Fusion. That notebook answered *"Why combine sparse and dense?"* — this one answers the harder follow-up questions:

* How do we **rigorously evaluate** whether the ensemble actually helps?
* How do we **tune the combination weights** instead of treating every ranker equally?
* When should we **skip the ensemble** and stick with a single retriever?

We will build a principled, evaluated ensemble from scratch, compare it head-to-head against individual retrievers, and use **Qwen3-8B** (4-bit) to verify that better retrieval translates to better generated answers.

## 🎯 Learning Objectives

By the end of this notebook you will be able to:

- **Explain why** single-method retrieval has blind spots and give concrete failure cases for both BM25 and dense retrieval
- **Implement** BM25, dense (FAISS), and Reciprocal Rank Fusion from scratch inside a clean `EnsembleRetriever` class
- **Derive** the RRF formula and reason about the role of the *k* parameter
- **Evaluate** retrieval quality with Recall@k and compare BM25 vs. dense vs. ensemble on a controlled test set
- **Tune ensemble weights** with a simple grid search and understand when weighted fusion outperforms uniform RRF

<div style="text-align: center;">

<img src="../images/ensemble_retrieval.svg" alt="Ensemble Retrieval" style="width:100%; height:auto;">
</div>

## §1 Why Single-Method Retrieval Has Blind Spots

### The keyword gap — dense retrieval's weakness

Dense retrievers project text into a continuous vector space where semantically similar passages cluster together. This is powerful, but it comes at a cost: the model has no mechanism to *enforce exact lexical matching*.

Consider a biomedical search for **"CRISPR-Cas9 off-target effects"**. A dense model may return passages about "gene-editing side-effects" — semantically related, but missing documents that use the precise term "CRISPR-Cas9." When a user types a jargon term, they often *need* exact-match results, and dense retrieval can silently drop them.

### The synonym gap — BM25's weakness

BM25 decomposes queries into tokens and scores documents by term frequency and inverse document frequency. It cannot know that "automobile" and "car" are the same concept, or that "myocardial infarction" and "heart attack" refer to the same medical event. If the query says "car" and the document says "automobile," BM25 assigns zero relevance for that token.

### An adversarial example

Imagine two passages in our knowledge base:

| Passage | BM25 finds it? | Dense finds it? |
|---------|:-:|:-:|
| *"CRISPR-Cas9 has revolutionised gene editing, but off-target mutations remain a concern."* | ✅ exact term match | ⚠️ might rank lower |
| *"Modern genome-modification tools occasionally alter unintended DNA regions."* | ❌ no keyword overlap | ✅ semantic match |

Neither retriever alone returns **both** passages. An ensemble does.

This is not a contrived edge case — it is the *default experience* whenever a corpus mixes formal and informal language, abbreviations and full forms, or domain jargon and layperson descriptions.

## §2 Retrieval Method Taxonomy

Retrieval methods fall into three broad families. Understanding their mechanics helps us predict when each will fail — and therefore when an ensemble adds value.

### Sparse retrieval (BM25 / TF-IDF)

Sparse methods represent documents as high-dimensional vectors where each dimension corresponds to a vocabulary token. Most values are zero (hence *sparse*). BM25 improves on raw TF-IDF by adding **saturation** (diminishing returns for repeated terms) and **document-length normalisation**, which prevents long documents from dominating results.

**Strengths:** blazing fast via inverted indexes, no GPU needed, excellent for exact-term recall, well-understood probabilistically.

**Weaknesses:** zero semantic understanding, sensitive to vocabulary mismatch, ignores word order.

### Dense retrieval

Dense methods use a neural encoder (typically a bi-encoder Transformer) to map both queries and documents to *d*-dimensional vectors (commonly 384–768 dims). Relevance is measured by cosine similarity or inner product in this learned space.

**Strengths:** captures paraphrases, synonyms, and conceptual similarity; works across languages when using multilingual encoders.

**Weaknesses:** requires GPU for encoding, index building is slower, can miss rare or novel terms not well-represented in training data.

### Hybrid / Ensemble approaches

Hybrid approaches combine the signals from both families. The simplest form is *late fusion*: run BM25 and dense retrieval independently, then merge the two ranked lists. More complex forms include *learned sparse* models (e.g., SPLADE) that inject neural semantics into sparse representations, and *ColBERT*-style late interaction that keeps per-token embeddings.

| Property | Sparse (BM25) | Dense (bi-encoder) | Ensemble |
|----------|:-:|:-:|:-:|
| Exact keyword recall | ★★★ | ★ | ★★★ |
| Semantic understanding | ★ | ★★★ | ★★★ |
| Latency | ★★★ | ★★ | ★★ |
| Complexity | ★ (simple) | ★★ | ★★★ |
| Robustness to query diversity | ★★ | ★★ | ★★★ |

## §3 Reciprocal Rank Fusion (RRF) — The Elegant Merging Algorithm

### The core idea

Given *m* rankers each producing a ranked list of documents, we want a single fused ranking. The naïve approach — normalise raw scores and sum them — is fragile because BM25 scores and cosine similarities live on entirely different scales and distributions.

**Reciprocal Rank Fusion** sidesteps score calibration altogether. It converts each ranker's output to *rank-based* contributions:

$$
\text{RRF}(d) = \sum_{r=1}^{m} \frac{1}{k + \text{rank}_r(d)}
$$

where $k$ is a smoothing constant (typically 60) and $\text{rank}_r(d)$ is the position of document $d$ in ranker $r$'s list (1-indexed; ∞ if absent).

### Why it works

1. **Rank-based:** immune to score scale differences — a rank of 1 always contributes $\frac{1}{k+1}$ regardless of the underlying score.
2. **Robust to outliers:** the $k$ parameter ensures no single top-1 ranking dominates the fused score. With $k=60$, rank 1 scores $1/61 \approx 0.0164$ while rank 2 scores $1/62 \approx 0.0161$ — a gentle slope, not a cliff.
3. **Democratic:** every ranker contributes equally, no weight tuning needed (though we'll add weights in §7).

### Worked example

Suppose BM25 and Dense each return their top-5 documents:

| Document | BM25 rank | Dense rank | RRF (k=60) |
|----------|:---------:|:----------:|:----------:|
| D₃ | 1 | 3 | 1/61 + 1/63 = **0.03227** |
| D₁ | 2 | 1 | 1/62 + 1/61 = **0.03244** ← winner |
| D₅ | 3 | 2 | 1/63 + 1/62 = **0.03199** |
| D₂ | 4 | — | 1/64 + 0 = 0.01563 |
| D₄ | — | 4 | 0 + 1/64 = 0.01563 |

D₁ wins because it appears near the top in *both* lists, even though D₃ holds BM25 rank 1. This consensus-based ranking is the key advantage.

### The role of *k*

Small *k* (e.g., 1) amplifies the gap between rank 1 and rank 2, behaving more like "take the top result." Large *k* (e.g., 1000) flattens the curve so all top-100 documents contribute almost equally. The default $k=60$ strikes a balance: top results still matter, but a document ranked 5th by two rankers beats a document ranked 1st by only one.

## §4 Setup

In [ ]:
!pip install -q rank_bm25 sentence-transformers faiss-cpu datasets transformers torch bitsandbytes accelerate

In [ ]:
import os, torch, warnings, numpy as np
from typing import Optional
warnings.filterwarnings("ignore")

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

# ── Cache paths (Google Drive for Colab persistence) ──
from google.colab import drive
drive.mount("/content/drive")

CACHE_DIR = "/content/drive/MyDrive/models"
os.makedirs(CACHE_DIR, exist_ok=True)

# ── Load Qwen3-8B in 4-bit ──
LLM_ID = "Qwen/Qwen3-8B"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(LLM_ID, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    LLM_ID,
    quantization_config=bnb_config,
    device_map="auto",
    cache_dir=CACHE_DIR,
)

# ── Load embedding model ──
EMBED_ID = "sentence-transformers/all-MiniLM-L6-v2"
embed_model = SentenceTransformer(EMBED_ID, cache_folder=CACHE_DIR)

print(f"✓ LLM loaded : {LLM_ID} (4-bit NF4)")
print(f"✓ Embeddings : {EMBED_ID} (dim={embed_model.get_sentence_embedding_dimension()})")
print(f"✓ Device     : {model.device}")

### Helper: LLM generation utility

We wrap tokenisation → generation → decoding into a single function so every downstream cell stays clean.

In [ ]:
def generate_answer(
    question: str,
    context: str,
    max_new_tokens: int = 512,
    temperature: float = 0.7,
) -> str:
    """Generate an answer using Qwen3-8B given a question and retrieved context."""
    system_msg = (
        "You are a helpful assistant. Answer the question using ONLY "
        "the provided context. If the context is insufficient, say so."
    )
    user_msg = f"Context:\n{context}\n\nQuestion: {question}"

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,          # Qwen3 specific
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
        )

    # Decode only the newly generated tokens
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

# Quick smoke test
print(generate_answer("What is 2+2?", "Basic arithmetic: 2+2 equals 4."))

## §5 Building the Knowledge Base

We deliberately craft ~20 passages so that some are best found by BM25 (rich in specific technical terms), others by dense retrieval (using synonyms/paraphrases), and a few by both. Each passage has an `id` we can reference in our evaluation set later.

In [ ]:
KNOWLEDGE_BASE: list[dict[str, str]] = [
    # ── Passages rich in specific keywords (BM25-friendly) ──
    {"id": "p01", "text": "CRISPR-Cas9 is a molecular tool that allows precise editing of DNA sequences. Off-target mutations remain a concern in therapeutic applications of CRISPR-Cas9."},
    {"id": "p02", "text": "The Python GIL (Global Interpreter Lock) prevents multiple native threads from executing Python bytecodes simultaneously, limiting true parallelism in CPython."},
    {"id": "p03", "text": "Kubernetes uses etcd as its backing store for all cluster data. Pods are the smallest deployable units in Kubernetes orchestration."},
    {"id": "p04", "text": "The Transformer architecture introduced by Vaswani et al. (2017) relies on multi-head self-attention and positional encoding to process sequences without recurrence."},
    {"id": "p05", "text": "PostgreSQL supports JSONB columns, full-text search with tsvector/tsquery, and advanced indexing strategies including GIN and GiST indexes."},
    {"id": "p06", "text": "TLS 1.3 reduced the handshake to a single round trip (1-RTT) and removed support for older cipher suites like RC4 and 3DES."},
    {"id": "p07", "text": "MapReduce is a programming model for processing large datasets in parallel across a Hadoop cluster using map and reduce functions."},

    # ── Passages using synonyms/paraphrases (dense-retrieval-friendly) ──
    {"id": "p08", "text": "Modern genome-modification tools occasionally alter unintended DNA regions, raising safety questions for clinical deployment."},
    {"id": "p09", "text": "Concurrent execution in the standard Python runtime is constrained by a mechanism that serialises thread access to the interpreter core."},
    {"id": "p10", "text": "Container orchestration platforms schedule workloads across a fleet of machines, restarting failed processes automatically."},
    {"id": "p11", "text": "Attention mechanisms let neural networks weigh the relevance of each input element relative to every other, enabling long-range dependency modelling."},
    {"id": "p12", "text": "Document-oriented databases can store nested structures and support searching within semi-structured records using specialised index types."},
    {"id": "p13", "text": "Encrypted connections between browsers and servers now establish secure channels faster, with obsolete cryptographic methods being phased out."},
    {"id": "p14", "text": "Distributed batch-processing frameworks split data into partitions, apply a function to each partition, and aggregate the results."},

    # ── Passages that both methods can find (overlap) ──
    {"id": "p15", "text": "Retrieval-Augmented Generation (RAG) combines a retriever that fetches relevant documents with a generator LLM that produces answers grounded in retrieved evidence."},
    {"id": "p16", "text": "BM25 is a probabilistic ranking function used by search engines to estimate the relevance of documents to a given keyword query."},
    {"id": "p17", "text": "Vector databases like FAISS and Milvus enable approximate nearest-neighbour search over high-dimensional embedding vectors."},
    {"id": "p18", "text": "Fine-tuning a pre-trained language model on domain-specific data often improves downstream task performance with limited labelled examples."},
    {"id": "p19", "text": "Gradient descent optimises neural network parameters by iteratively adjusting weights in the direction that minimises the loss function."},
    {"id": "p20", "text": "Reinforcement learning from human feedback (RLHF) aligns language models with human preferences by training a reward model on pairwise comparisons."},
]

passages = [p["text"] for p in KNOWLEDGE_BASE]
passage_ids = [p["id"] for p in KNOWLEDGE_BASE]
print(f"Knowledge base: {len(KNOWLEDGE_BASE)} passages loaded.")

### BM25 Retriever

We tokenize each passage into lowercased words and build a BM25 index using `rank_bm25`.

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(text: str) -> list[str]:
    """Simple whitespace + lowercase tokenizer."""
    return text.lower().split()

tokenized_corpus = [tokenize(p) for p in passages]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_retrieve(query: str, top_k: int = 10) -> list[tuple[str, float]]:
    """Return (passage_id, bm25_score) pairs, sorted by score descending."""
    scores = bm25.get_scores(tokenize(query))
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(passage_ids[i], float(scores[i])) for i in top_indices]

# Demo
print("BM25 top-5 for 'CRISPR-Cas9 off-target mutations':")
for pid, score in bm25_retrieve("CRISPR-Cas9 off-target mutations", top_k=5):
    print(f"  {pid} | score={score:.4f} | {passages[passage_ids.index(pid)][:80]}…")

### Dense Retriever with FAISS

We encode every passage once with our bi-encoder, build a FAISS inner-product index, and search by encoding the query at retrieval time.

In [ ]:
import faiss

# Encode corpus once
corpus_embeddings = embed_model.encode(passages, normalize_embeddings=True, show_progress_bar=True)
corpus_embeddings = np.array(corpus_embeddings, dtype="float32")

# Build FAISS index (inner product ≈ cosine similarity when vectors are normalised)
dim = corpus_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(corpus_embeddings)
print(f"FAISS index built: {faiss_index.ntotal} vectors, dim={dim}")

def dense_retrieve(query: str, top_k: int = 10) -> list[tuple[str, float]]:
    """Return (passage_id, similarity_score) pairs, sorted by score descending."""
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = faiss_index.search(q_emb, top_k)
    return [(passage_ids[int(i)], float(s)) for s, i in zip(scores[0], indices[0])]

# Demo
print("\nDense top-5 for 'CRISPR-Cas9 off-target mutations':")
for pid, score in dense_retrieve("CRISPR-Cas9 off-target mutations", top_k=5):
    print(f"  {pid} | score={score:.4f} | {passages[passage_ids.index(pid)][:80]}…")

### Demonstrating the blind spots

Let's run two queries designed to expose each retriever's weakness:

1. **Keyword-heavy query:** *"Python GIL bytecodes CPython"* — BM25 should excel, dense may struggle.
2. **Semantic query:** *"What limits running code at the same time in Python?"* — dense should find the paraphrase (p09), BM25 may miss it.

In our knowledge base, query 2 maps to the concept of concurrency constraints (p02/p09). BM25 can match p02 via the word "Python" but will likely miss p09 entirely since it uses no shared keywords.

In [ ]:
query_keyword = "Python GIL bytecodes CPython"
query_semantic = "What limits running code at the same time in Python?"

print("=== BM25 results ===")
for q in [query_keyword, query_semantic]:
    top = bm25_retrieve(q, top_k=3)
    print(f"\nQuery: {q}")
    for pid, s in top:
        print(f"  {pid} ({s:.3f})")

print("\n=== Dense results ===")
for q in [query_keyword, query_semantic]:
    top = dense_retrieve(q, top_k=3)
    print(f"\nQuery: {q}")
    for pid, s in top:
        print(f"  {pid} ({s:.3f})")

## §6 Implementation — The Ensemble Retriever

### Reciprocal Rank Fusion function

In [ ]:
def reciprocal_rank_fusion(
    ranked_lists: list[list[str]],
    k: int = 60,
) -> list[tuple[str, float]]:
    """
    Merge multiple ranked lists of document IDs using RRF.

    Parameters
    ----------
    ranked_lists : list of lists of passage IDs, each ordered by relevance.
    k            : smoothing constant (default 60).

    Returns
    -------
    List of (passage_id, rrf_score) sorted by fused score descending.
    """
    rrf_scores: dict[str, float] = {}
    for ranked in ranked_lists:
        for rank, doc_id in enumerate(ranked, start=1):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

# Quick sanity check with the worked example from §3
bm25_ranking = ["D3", "D1", "D5", "D2"]
dense_ranking = ["D1", "D5", "D3", "D4"]
fused = reciprocal_rank_fusion([bm25_ranking, dense_ranking], k=60)
print("Worked-example RRF results:")
for doc, score in fused:
    print(f"  {doc}: {score:.5f}")

### The `EnsembleRetriever` class

This class encapsulates BM25, dense, and RRF fusion into a clean interface.

In [ ]:
class EnsembleRetriever:
    """Combines BM25 and dense retrieval via Reciprocal Rank Fusion."""

    def __init__(
        self,
        bm25_index: BM25Okapi,
        faiss_index: faiss.IndexFlatIP,
        embed_model: SentenceTransformer,
        passages: list[str],
        passage_ids: list[str],
        k: int = 60,
    ):
        self.bm25 = bm25_index
        self.faiss_index = faiss_index
        self.embed_model = embed_model
        self.passages = passages
        self.passage_ids = passage_ids
        self.k = k

    def _bm25_rank(self, query: str, top_k: int) -> list[str]:
        scores = self.bm25.get_scores(tokenize(query))
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [self.passage_ids[i] for i in top_idx]

    def _dense_rank(self, query: str, top_k: int) -> list[str]:
        q_emb = self.embed_model.encode([query], normalize_embeddings=True).astype("float32")
        _, indices = self.faiss_index.search(q_emb, top_k)
        return [self.passage_ids[int(i)] for i in indices[0]]

    def retrieve(
        self, query: str, top_k: int = 5, pool_size: int = 20
    ) -> list[tuple[str, float, str]]:
        """
        Return top_k (passage_id, rrf_score, passage_text) tuples.
        pool_size controls how many candidates each sub-retriever contributes.
        """
        bm25_ranked = self._bm25_rank(query, pool_size)
        dense_ranked = self._dense_rank(query, pool_size)
        fused = reciprocal_rank_fusion([bm25_ranked, dense_ranked], self.k)

        results = []
        for pid, score in fused[:top_k]:
            idx = self.passage_ids.index(pid)
            results.append((pid, score, self.passages[idx]))
        return results

ensemble = EnsembleRetriever(bm25, faiss_index, embed_model, passages, passage_ids)

# Demo query
print("Ensemble top-5 for 'CRISPR-Cas9 off-target mutations':\n")
for pid, score, text in ensemble.retrieve("CRISPR-Cas9 off-target mutations"):
    print(f"  {pid} | RRF={score:.5f}")
    print(f"       {text[:90]}…\n")

### Side-by-side comparison on multiple queries

Let's compare all three retrievers on queries that stress different capabilities.

In [ ]:
test_queries = [
    "CRISPR-Cas9 off-target mutations",
    "What limits running code at the same time in Python?",
    "How does the Transformer architecture handle sequences?",
    "distributed data processing frameworks",
    "securing browser-server communication",
]

for q in test_queries:
    bm25_top = [pid for pid, _ in bm25_retrieve(q, top_k=3)]
    dense_top = [pid for pid, _ in dense_retrieve(q, top_k=3)]
    ens_top = [pid for pid, _, _ in ensemble.retrieve(q, top_k=3)]
    print(f"Query: {q}")
    print(f"  BM25   : {bm25_top}")
    print(f"  Dense  : {dense_top}")
    print(f"  Ensemble: {ens_top}\n")

## §7 Weighted Ensembles — Beyond Uniform RRF

Standard RRF treats every ranker equally. But what if we know from experience that our dense retriever is generally more reliable on our corpus? We can introduce **weights**:

$$
\text{WRRF}(d) = \sum_{r=1}^{m} w_r \cdot \frac{1}{k + \text{rank}_r(d)}
$$

where $w_r$ is the weight for ranker $r$ and $\sum w_r = 1$. Setting all weights to $1/m$ recovers standard RRF.

Alternatively we can do **score-level fusion**: normalise BM25 and dense scores to [0, 1], then compute a weighted sum:

$$
\text{score}(d) = w_{\text{dense}} \cdot \hat{s}_{\text{dense}}(d) + w_{\text{sparse}} \cdot \hat{s}_{\text{sparse}}(d)
$$

We'll implement the score-level variant and tune weights with a grid search.

In [ ]:
def min_max_normalise(scores: list[float]) -> list[float]:
    """Normalise scores to [0, 1] via min-max scaling."""
    lo, hi = min(scores), max(scores)
    if hi - lo < 1e-9:
        return [0.5] * len(scores)
    return [(s - lo) / (hi - lo) for s in scores]


def weighted_fusion(
    query: str,
    w_dense: float = 0.5,
    top_k: int = 5,
    pool_size: int = 20,
) -> list[tuple[str, float, str]]:
    """
    Score-level weighted fusion of BM25 and dense retrieval.
    w_dense in [0, 1]; w_sparse = 1 - w_dense.
    """
    w_sparse = 1.0 - w_dense

    # BM25 scores
    bm25_scores_raw = bm25.get_scores(tokenize(query))
    bm25_top_idx = np.argsort(bm25_scores_raw)[::-1][:pool_size]
    bm25_candidates = {passage_ids[i]: bm25_scores_raw[i] for i in bm25_top_idx}

    # Dense scores
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    dense_scores_raw, dense_indices = faiss_index.search(q_emb, pool_size)
    dense_candidates = {
        passage_ids[int(i)]: float(s) for s, i in zip(dense_scores_raw[0], dense_indices[0])
    }

    # Union of candidates
    all_ids = list(set(bm25_candidates) | set(dense_candidates))
    bm25_vals = [bm25_candidates.get(pid, 0.0) for pid in all_ids]
    dense_vals = [dense_candidates.get(pid, 0.0) for pid in all_ids]

    bm25_norm = min_max_normalise(bm25_vals)
    dense_norm = min_max_normalise(dense_vals)

    fused: list[tuple[str, float]] = []
    for pid, bn, dn in zip(all_ids, bm25_norm, dense_norm):
        fused.append((pid, w_sparse * bn + w_dense * dn))

    fused.sort(key=lambda x: x[1], reverse=True)
    results = []
    for pid, score in fused[:top_k]:
        idx = passage_ids.index(pid)
        results.append((pid, score, passages[idx]))
    return results

# Demo
print("Weighted fusion (w_dense=0.6) for 'securing browser-server communication':\n")
for pid, score, text in weighted_fusion("securing browser-server communication", w_dense=0.6):
    print(f"  {pid} | score={score:.4f} | {text[:80]}…")

### Tuning weights with grid search

We define a small evaluation set: queries paired with known relevant passage IDs. We sweep `w_dense` from 0 to 1 and pick the weight that maximises average Recall@5.

In [ ]:
EVAL_SET: list[dict] = [
    {"query": "CRISPR-Cas9 off-target mutations",
     "relevant": ["p01", "p08"]},
    {"query": "What limits running code at the same time in Python?",
     "relevant": ["p02", "p09"]},
    {"query": "How does the Transformer architecture handle sequences?",
     "relevant": ["p04", "p11"]},
    {"query": "distributed data processing frameworks",
     "relevant": ["p07", "p14"]},
    {"query": "securing browser-server communication",
     "relevant": ["p06", "p13"]},
    {"query": "searching within semi-structured database records",
     "relevant": ["p05", "p12"]},
    {"query": "retrieval augmented generation for LLMs",
     "relevant": ["p15", "p16", "p17"]},
    {"query": "aligning language models with human preferences",
     "relevant": ["p18", "p20"]},
]

def recall_at_k(retrieved_ids: list[str], relevant_ids: list[str], k: int = 5) -> float:
    """Fraction of relevant documents found in the top-k retrieved."""
    top_k = set(retrieved_ids[:k])
    return len(top_k & set(relevant_ids)) / len(relevant_ids)

# Grid search over w_dense
best_w, best_recall = 0.0, 0.0
results_by_w = {}

for w_dense_pct in range(0, 105, 5):
    w_dense = w_dense_pct / 100.0
    recalls = []
    for item in EVAL_SET:
        retrieved = weighted_fusion(item["query"], w_dense=w_dense, top_k=5)
        ret_ids = [pid for pid, _, _ in retrieved]
        recalls.append(recall_at_k(ret_ids, item["relevant"]))
    avg = np.mean(recalls)
    results_by_w[w_dense] = avg
    if avg > best_recall:
        best_recall = avg
        best_w = w_dense

print(f"Best w_dense = {best_w:.2f} → avg Recall@5 = {best_recall:.3f}\n")
print("Weight sweep:")
for w, r in sorted(results_by_w.items()):
    bar = "█" * int(r * 40)
    print(f"  w_dense={w:.2f} | R@5={r:.3f} | {bar}")

## §8 Evaluation — BM25 vs Dense vs Ensemble

Now let's rigorously compare **BM25 alone**, **dense alone**, **RRF ensemble**, and **weighted fusion** (at the best weight found above) on our evaluation set.

In [ ]:
from collections import defaultdict

method_recalls: dict[str, list[float]] = defaultdict(list)

for item in EVAL_SET:
    q = item["query"]
    rel = item["relevant"]

    # BM25
    bm25_ids = [pid for pid, _ in bm25_retrieve(q, top_k=5)]
    method_recalls["BM25"].append(recall_at_k(bm25_ids, rel))

    # Dense
    dense_ids = [pid for pid, _ in dense_retrieve(q, top_k=5)]
    method_recalls["Dense"].append(recall_at_k(dense_ids, rel))

    # RRF Ensemble
    ens_ids = [pid for pid, _, _ in ensemble.retrieve(q, top_k=5)]
    method_recalls["RRF Ensemble"].append(recall_at_k(ens_ids, rel))

    # Weighted Fusion (best weight)
    wf_ids = [pid for pid, _, _ in weighted_fusion(q, w_dense=best_w, top_k=5)]
    method_recalls["Weighted Fusion"].append(recall_at_k(wf_ids, rel))

print(f"{'Method':<20} {'Avg R@5':>8}  {'Per-query R@5'}")
print("-" * 70)
for method, recalls in method_recalls.items():
    per_q = "  ".join(f"{r:.2f}" for r in recalls)
    print(f"{method:<20} {np.mean(recalls):>8.3f}  [{per_q}]")

### Interpreting the results

You should observe that:

1. **BM25** excels on keyword-heavy queries (e.g., "CRISPR-Cas9 off-target mutations") but falters on paraphrases.
2. **Dense** excels on semantic queries but can miss passages phrased with exact technical terms.
3. **RRF Ensemble** combines the strengths, typically matching or exceeding both individual retrievers.
4. **Weighted Fusion** with a tuned weight can squeeze out additional gains — the grid search reveals the optimal blend for your specific corpus and query distribution.

This pattern — *the ensemble is at least as good as the best individual retriever, and often better* — is the fundamental reason ensembles dominate production RAG systems.

### LLM Answer Quality Comparison

Better retrieval should produce better answers. Let's verify by feeding each retriever's results to Qwen3-8B and inspecting the generated answers side by side.

In [ ]:
comparison_query = "What are the safety concerns with modern gene-editing tools?"

print(f"Query: {comparison_query}")
print("=" * 60 + "\n")

# Gather context from each retriever
for label, retrieve_fn in [
    ("BM25", lambda q: [(pid, passages[passage_ids.index(pid)])
                         for pid, _ in bm25_retrieve(q, top_k=3)]),
    ("Dense", lambda q: [(pid, passages[passage_ids.index(pid)])
                          for pid, _ in dense_retrieve(q, top_k=3)]),
    ("Ensemble (RRF)", lambda q: [(pid, text)
                                    for pid, _, text in ensemble.retrieve(q, top_k=3)]),
]:
    results = retrieve_fn(comparison_query)
    context = "\n\n".join(f"[{pid}] {text}" for pid, text in results)
    print(f"--- {label} ---")
    print(f"Retrieved: {[pid for pid, _ in results]}")
    answer = generate_answer(comparison_query, context)
    print(f"Answer: {answer}\n")

## §9 When to Use Ensembles vs. a Single Retriever

Ensembles are not free — they add **latency** (two retrieval passes instead of one) and **complexity** (more moving parts to monitor and debug). Here is a decision framework:

### Use an ensemble when:
- Your **query distribution is diverse** — some users type keywords, others ask natural-language questions.
- Your **corpus mixes registers** — formal papers, informal documentation, code snippets, user forums.
- You need **high recall** — missing a relevant document is more costly than slightly slower retrieval (e.g., medical, legal, compliance domains).
- You have the **infrastructure** to run two retrievers in parallel (the latency overhead then shrinks to max(BM25, dense) instead of sum).

### Stick with a single retriever when:
- Queries are **homogeneous** — e.g., all natural-language questions on a customer-support bot (dense is likely sufficient).
- Latency is **ultra-critical** (sub-10 ms) — BM25 alone on an inverted index is hard to beat.
- Your corpus is **small** (<1000 documents) — the overlap between BM25 and dense top-k results will be high, and the ensemble adds little.

### Cost-benefit summary

| Factor | Single retriever | Ensemble |
|--------|:---:|:---:|
| Latency | Lower | Higher (can be mitigated with parallelism) |
| Recall | Good for its niche | Consistently high across query types |
| Maintenance | Simpler | More complex (two indexes, two models) |
| Best for | Homogeneous queries, latency-critical | Diverse queries, recall-critical |

## §10 Connection to `fusion_retrieval.ipynb`

This notebook extends the concepts introduced in **[fusion_retrieval.ipynb](fusion_retrieval.ipynb)**, which covers:

- The *foundational intuition* for why combining sparse and dense retrieval matters.
- A first implementation of BM25 + dense search with RRF.
- A complete RAG pipeline using the fused retriever.

**What this notebook adds:**

| fusion_retrieval.ipynb | ensemble_retrieval.ipynb (this notebook) |
|:-:|:-:|
| Introduces BM25 + Dense + RRF | Derives RRF from first principles, explains the *k* parameter |
| Single demo pipeline | Formal evaluation with Recall@k across multiple queries |
| Fixed equal weights | Weighted ensembles with grid-search tuning |
| One retrieval comparison | Systematic blind-spot analysis for each retriever |
| — | Decision framework for when ensembles help vs. hurt |

Think of `fusion_retrieval.ipynb` as "your first hybrid retriever" and this notebook as "making your hybrid retriever production-ready."

## §11 Advanced — Sensitivity to the RRF *k* Parameter

Let's empirically measure how the choice of *k* affects ensemble quality.

In [ ]:
class EnsembleRetrieverK(EnsembleRetriever):
    """EnsembleRetriever variant that accepts k at query time."""
    def retrieve_with_k(
        self, query: str, k: int, top_k: int = 5, pool_size: int = 20
    ) -> list[tuple[str, float, str]]:
        bm25_ranked = self._bm25_rank(query, pool_size)
        dense_ranked = self._dense_rank(query, pool_size)
        fused = reciprocal_rank_fusion([bm25_ranked, dense_ranked], k=k)
        results = []
        for pid, score in fused[:top_k]:
            idx = self.passage_ids.index(pid)
            results.append((pid, score, self.passages[idx]))
        return results

ens_k = EnsembleRetrieverK(bm25, faiss_index, embed_model, passages, passage_ids)

print(f"{'k':>6} | {'Avg R@5':>8}")
print("-" * 20)
for k_val in [1, 5, 10, 20, 40, 60, 100, 200, 500]:
    recalls = []
    for item in EVAL_SET:
        ret = ens_k.retrieve_with_k(item["query"], k=k_val, top_k=5)
        ret_ids = [pid for pid, _, _ in ret]
        recalls.append(recall_at_k(ret_ids, item["relevant"]))
    print(f"{k_val:>6} | {np.mean(recalls):>8.3f}")

The table above should show that very small *k* (≤5) can be volatile — the fusion becomes too sensitive to exact rank positions. Very large *k* (≥200) flattens differences. The sweet spot is typically in the range **20–100**, which is why $k=60$ is the standard default in the literature (Cormack et al., 2009).

## 📝 Exercises

### Exercise 1: Add a Third Ranker

Extend the `EnsembleRetriever` to accept a **TF-IDF** retriever as a third ranker alongside BM25 and dense. Use `sklearn.feature_extraction.text.TfidfVectorizer` to build the TF-IDF index. Feed all three ranked lists into `reciprocal_rank_fusion` and measure whether Recall@5 improves.

*Hint:* TF-IDF and BM25 are both sparse methods, so the improvement might be marginal — but the exercise teaches you how to plug in additional rankers.

In [ ]:
# Exercise 1 — Starter code
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity
#
# tfidf = TfidfVectorizer()
# tfidf_matrix = tfidf.fit_transform(passages)
#
# def tfidf_retrieve(query: str, top_k: int = 10) -> list[str]:
#     q_vec = tfidf.transform([query])
#     sims = cosine_similarity(q_vec, tfidf_matrix).flatten()
#     top_idx = np.argsort(sims)[::-1][:top_k]
#     return [passage_ids[i] for i in top_idx]
#
# # TODO: Integrate into EnsembleRetriever and run evaluation
# pass

### Exercise 2: Query-Dependent Weight Selection

Instead of a single global weight for the entire corpus, implement a system that selects the dense-vs-sparse weight **per query** based on a simple heuristic: if the query contains rare technical terms (words with high IDF), upweight BM25; if it is a natural-language question, upweight dense.

*Hint:* Compute the average IDF of query tokens. If it exceeds a threshold, set `w_dense = 0.3`; otherwise set `w_dense = 0.7`.

In [ ]:
# Exercise 2 — Starter code
# import math
#
# # Build IDF dictionary from the BM25 corpus
# doc_count = len(tokenized_corpus)
# df = {}  # document frequency
# for doc in tokenized_corpus:
#     for token in set(doc):
#         df[token] = df.get(token, 0) + 1
# idf = {t: math.log((doc_count - f + 0.5) / (f + 0.5) + 1) for t, f in df.items()}
#
# def adaptive_weight(query: str, idf_threshold: float = 3.0) -> float:
#     tokens = tokenize(query)
#     avg_idf = np.mean([idf.get(t, 0.0) for t in tokens]) if tokens else 0.0
#     return 0.3 if avg_idf > idf_threshold else 0.7
#
# # TODO: Use adaptive_weight(query) as w_dense in weighted_fusion
# pass

### Exercise 3: Recall@k Curves

Plot Recall@k for k ∈ {1, 2, 3, 5, 10, 15, 20} for BM25, dense, and ensemble. Use `matplotlib` to create a line chart. This visualises how quickly each method surfaces all relevant documents as *k* grows.

In [ ]:
# Exercise 3 — Starter code
# import matplotlib.pyplot as plt
#
# k_values = [1, 2, 3, 5, 10, 15, 20]
# methods = {
#     "BM25": lambda q, k: [pid for pid, _ in bm25_retrieve(q, top_k=k)],
#     "Dense": lambda q, k: [pid for pid, _ in dense_retrieve(q, top_k=k)],
#     "Ensemble": lambda q, k: [pid for pid, _, _ in ensemble.retrieve(q, top_k=k)],
# }
#
# for name, fn in methods.items():
#     avg_recalls = []
#     for k in k_values:
#         recalls = []
#         for item in EVAL_SET:
#             ret_ids = fn(item["query"], k)
#             recalls.append(recall_at_k(ret_ids, item["relevant"], k=k))
#         avg_recalls.append(np.mean(recalls))
#     plt.plot(k_values, avg_recalls, marker="o", label=name)
#
# plt.xlabel("k")
# plt.ylabel("Recall@k")
# plt.title("Recall@k Comparison")
# plt.legend()
# plt.grid(True, alpha=0.3)
# plt.show()

## 🔑 Key Takeaways

1. **No single retriever is best for all queries.** BM25 wins on exact-term queries; dense wins on paraphrase/semantic queries. An ensemble hedges against both failure modes.

2. **Reciprocal Rank Fusion is elegant and parameter-light.** By converting raw scores to rank-based contributions, RRF avoids the score-calibration problem entirely. The only hyper-parameter, *k*, is robust across a wide range (20–100).

3. **Weighted fusion can outperform uniform RRF** when you have even a small labelled evaluation set. A simple grid search over a single weight parameter is often enough.

4. **Evaluation drives good decisions.** Without measuring Recall@k on representative queries, you're guessing. Always build a small eval set before choosing your retrieval strategy.

5. **Ensembles add complexity.** They are most valuable when queries are diverse and recall is critical. For homogeneous, latency-sensitive workloads, a well-tuned single retriever may suffice.

## 📚 References

1. **Robertson, S. & Zaragoza, H.** (2009). *The Probabilistic Relevance Framework: BM25 and Beyond*. Foundations and Trends in IR.
2. **Cormack, G. V., Clarke, C. L. A., & Buettcher, S.** (2009). *Reciprocal Rank Fusion outperforms Condorcet and individual Rank Learning Methods*. SIGIR.
3. **Karpukhin, V. et al.** (2020). *Dense Passage Retrieval for Open-Domain Question Answering*. EMNLP.
4. **Lewis, P. et al.** (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. NeurIPS.
5. **Ma, X. et al.** (2023). *Fine-Tuning LLaMA for Multi-Stage Text Retrieval*. arXiv:2310.08319.
6. **Vaswani, A. et al.** (2017). *Attention Is All You Need*. NeurIPS.
7. **sentence-transformers documentation:** https://www.sbert.net/
8. **FAISS documentation:** https://github.com/facebookresearch/faiss